# Notebook 04: Bilingual Training Data Preparation

## Objective

Prepare the provisional English-Edo dataset for intent classification.

The notebook will:

1. Load the provisional bilingual dataset.
2. Inspect translation completeness.
3. Reuse the existing group-aware train, validation, and test splits.
4. Expand each record into English and Edo question examples.
5. Preserve record IDs and split groups.
6. Validate the resulting model-ready dataset.

## Important considerations

- English and Edo versions of the same question must remain in the same split.
- Machine-generated Edo translations are provisional.
- Answers are not classifier inputs.
- Agricultural accuracy and translation quality have not been independently verified.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_project_root(start_path=Path.cwd()):
    for path in [start_path, *start_path.parents]:
        if (
            (path / "data" / "processed").is_dir()
            and (path / "notebooks").is_dir()
        ):
            return path

    raise FileNotFoundError(
        "Could not locate the EdoFarm AI project root."
    )

ROOT = find_project_root()
DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
METADATA_DIR = DATA_DIR / "metadata"

BILINGUAL_PATH = (
    PROCESSED_DIR /
    "edofarm_bilingual_dataset_provisional_v1.csv"
)

print("Project root:", ROOT)
print("Bilingual dataset exists:", BILINGUAL_PATH.exists())

if not BILINGUAL_PATH.exists():
    raise FileNotFoundError(BILINGUAL_PATH)

Project root: c:\Users\XXMUGENX\3D Objects\edofarm-ai
Bilingual dataset exists: True


## 1. Load the provisional bilingual dataset

Load the exported dataset and verify that its identifiers,
English questions, Edo questions, and intent labels are present.

In [2]:
bilingual_df = pd.read_csv(BILINGUAL_PATH)

required_columns = [
    "record_id",
    "translation_group_id",
    "split_group_id",
    "english_question",
    "edo_question_final",
    "primary_intent",
    "translation_unit_type"
]

missing_columns = [
    column for column in required_columns
    if column not in bilingual_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Dataset shape:", bilingual_df.shape)
print("Unique records:", bilingual_df["record_id"].nunique())
print("Unique translation groups:", bilingual_df["translation_group_id"].nunique())

print("\nIntent distribution:")
display(
    bilingual_df["primary_intent"]
    .value_counts()
    .to_frame("records")
)

print("\nTranslation completeness:")
print(
    "English questions:",
    bilingual_df["english_question"].notna().sum()
)
print(
    "Edo questions:",
    bilingual_df["edo_question_final"].notna().sum()
)

display(
    bilingual_df[
        [
            "record_id",
            "english_question",
            "edo_question_final",
            "primary_intent"
        ]
    ].head(5)
)

Dataset shape: (306, 35)
Unique records: 306
Unique translation groups: 306

Intent distribution:


,records
primary_intent,
out_of_scope,75
soil_management,44
disease_management,40
fertilizer_application,40
harvesting,33
irrigation,32
pest_management,25
crop_cultivation,17



Translation completeness:
English questions: 306
Edo questions: 306


,record_id,english_question,edo_question_final,primary_intent
0,AQA_00009,which diseases can be spread by whiteflies?,Emianmwen vbe na nẹ whiteflies sẹtin kpe vbe e...,disease_management
1,AQA_00027,why is Integrated Pest Management used?,Vbọ zẹ nẹ a na lọghọ Integrated Pest Management?,pest_management
2,AQA_00031,What is the best way to store harvested wheat ...,De odẹ nọ maan sẹ nẹ a na sẹ wheat nẹ a kọkọ v...,harvesting
3,AQA_00045,How should the soil be prepared before plantin...,De vbene a khẹke nẹ a ru otọ hẹ kevbe nẹ a kọ ...,crop_cultivation
4,AQA_00061,how can pest and diseases be managed in maize ...,De vbene a sẹtin gbe erhanmwen kevbe emianmwen...,pest_management


## 2. Locate existing dataset splits

Search the data directory for previously exported training,
validation, and test CSV files.

These files should contain the original record IDs and intent labels.
We will use their record membership to assign both language versions
to the same split.

In [3]:
split_keywords = ("train", "validation", "valid", "val", "test")

split_candidates = []

for path in DATA_DIR.rglob("*.csv"):
    if "raw" in path.parts:
        continue

    filename = path.stem.lower()

    if any(keyword in filename for keyword in split_keywords):
        try:
            columns = list(pd.read_csv(path, nrows=0).columns)

            if "record_id" in columns:
                split_candidates.append({
                    "file": str(path.relative_to(ROOT)),
                    "columns": columns
                })

        except Exception as error:
            print(f"Could not inspect {path.name}: {error}")

if split_candidates:
    for candidate in split_candidates:
        print("\nFile:", candidate["file"])
        print("Columns:", candidate["columns"])
else:
    print("No existing split CSV files were detected.")


File: data\processed\classifier_test.csv
Columns: ['record_id', 'question', 'primary_intent', 'split_group_id', 'near_duplicate_group', 'source_dataset', 'review_decision', 'answer_quality']

File: data\processed\classifier_train.csv
Columns: ['record_id', 'question', 'primary_intent', 'split_group_id', 'near_duplicate_group', 'source_dataset', 'review_decision', 'answer_quality']

File: data\processed\classifier_validation.csv
Columns: ['record_id', 'question', 'primary_intent', 'split_group_id', 'near_duplicate_group', 'source_dataset', 'review_decision', 'answer_quality']

File: data\processed\edofarm_training_dataset.csv
Columns: ['record_id', 'source_dataset', 'source_record_id', 'question', 'answer', 'crop', 'country', 'state_region', 'local_area', 'source_date', 'matched_intents_v3', 'relevance_status_v3', 'exact_question_duplicate_count', 'candidate_intents', 'proposed_primary_intent', 'primary_intent', 'review_decision', 'answer_quality', 'agricultural_accuracy', 'nigerian_re

## 2.1 Load and validate existing splits

Load the training, validation, and test datasets created in Notebook 02.

Validation checks:

1. Every record appears in exactly one split.
2. All 306 bilingual records are represented.
3. English and Edo versions inherit the same split.
4. Duplicate groups are not distributed across different splits.
5. Intent labels and split group IDs match the bilingual dataset.

In [4]:
split_files = {
    "train": PROCESSED_DIR / "classifier_train.csv",
    "validation": PROCESSED_DIR / "classifier_validation.csv",
    "test": PROCESSED_DIR / "classifier_test.csv"
}

split_frames = []

for split_name, file_path in split_files.items():

    if not file_path.exists():
        raise FileNotFoundError(file_path)

    split_df = pd.read_csv(file_path)

    required_split_columns = [
        "record_id",
        "primary_intent",
        "split_group_id"
    ]

    missing = [
        col for col in required_split_columns
        if col not in split_df.columns
    ]

    if missing:
        raise ValueError(
            f"{split_name} is missing columns: {missing}"
        )

    split_df = split_df[required_split_columns].copy()

    split_df["record_id"] = (
        split_df["record_id"].astype(str).str.strip()
    )

    split_df["split_group_id"] = (
        split_df["split_group_id"].astype(str).str.strip()
    )

    if split_df["record_id"].duplicated().any():
        raise ValueError(
            f"Duplicate record IDs found in {split_name}."
        )

    split_df["split"] = split_name
    split_frames.append(split_df)


split_membership = pd.concat(
    split_frames,
    ignore_index=True
)

# Normalize IDs in the bilingual dataset.
bilingual_df["record_id"] = (
    bilingual_df["record_id"].astype(str).str.strip()
)

bilingual_df["split_group_id"] = (
    bilingual_df["split_group_id"].astype(str).str.strip()
)

# Ensure records do not appear in multiple splits.
if split_membership["record_id"].duplicated().any():
    raise ValueError(
        "Some records appear in more than one split."
    )

bilingual_ids = set(bilingual_df["record_id"])
split_ids = set(split_membership["record_id"])

missing_from_splits = bilingual_ids - split_ids
extra_in_splits = split_ids - bilingual_ids

print("Bilingual records:", len(bilingual_ids))
print("Records in existing splits:", len(split_ids))
print("Missing from splits:", len(missing_from_splits))
print("Extra records in splits:", len(extra_in_splits))

if missing_from_splits or extra_in_splits:
    raise ValueError(
        "The bilingual dataset and existing splits do not contain "
        "exactly the same records."
    )

# Check group IDs and intent labels.
comparison = bilingual_df[
    ["record_id", "split_group_id", "primary_intent"]
].merge(
    split_membership,
    on="record_id",
    how="inner",
    suffixes=("_bilingual", "_split"),
    validate="one_to_one"
)

group_mismatch = comparison[
    comparison["split_group_id_bilingual"]
    != comparison["split_group_id_split"]
]

intent_mismatch = comparison[
    comparison["primary_intent_bilingual"]
    != comparison["primary_intent_split"]
]

if not group_mismatch.empty:
    raise ValueError("Split group IDs do not match.")

if not intent_mismatch.empty:
    raise ValueError("Intent labels do not match.")

# Verify that duplicate groups do not cross splits.
group_split_counts = (
    split_membership.groupby("split_group_id")["split"]
    .nunique()
)

if (group_split_counts > 1).any():
    raise ValueError(
        "A duplicate group appears across multiple splits."
    )

# Attach the original split to each bilingual record.
split_map = split_membership.set_index("record_id")["split"]

bilingual_df["split"] = bilingual_df["record_id"].map(split_map)

if bilingual_df["split"].isna().any():
    raise ValueError("Some records have no assigned split.")

print("\nSPLIT VALIDATION PASSED")
print("-----------------------")

print("\nRecord distribution:")
print(bilingual_df["split"].value_counts())

print("\nUnique duplicate groups per split:")
print(
    bilingual_df.groupby("split")["split_group_id"].nunique()
)

print("\nIntent distribution by split:")
display(
    pd.crosstab(
        bilingual_df["split"],
        bilingual_df["primary_intent"]
    )
)

Bilingual records: 306
Records in existing splits: 306
Missing from splits: 0
Extra records in splits: 0

SPLIT VALIDATION PASSED
-----------------------

Record distribution:
split
train         196
test           61
validation     49
Name: count, dtype: int64

Unique duplicate groups per split:
split
test           56
train         185
validation     46
Name: split_group_id, dtype: int64

Intent distribution by split:


primary_intent,crop_cultivation,disease_management,fertilizer_application,harvesting,irrigation,out_of_scope,pest_management,soil_management
split,,,,,,,,
test,3,8,8,6,7,15,5,9
train,11,25,26,22,20,48,16,28
validation,3,7,6,5,5,12,4,7


## 3. Expand records into bilingual classifier examples

Each underlying record will produce two classifier examples:

1. English: the original English question.
2. Edo (Bini): the provisional machine-generated Edo translation.

Both examples will retain:
- The same record ID.
- The same intent label.
- The same duplicate group.
- The same train/validation/test assignment.

Machine-generated Edo examples will be explicitly marked as provisional.

The answer fields will not be included in the classifier dataset.

In [5]:
# Confirm that all questions have usable text.

english_missing = (
    bilingual_df["english_question"].isna()
    | bilingual_df["english_question"].astype(str).str.strip().eq("")
)

edo_missing = (
    bilingual_df["edo_question_final"].isna()
    | bilingual_df["edo_question_final"].astype(str).str.strip().eq("")
)

if english_missing.any():
    raise ValueError(
        f"{english_missing.sum()} English questions are missing."
    )

if edo_missing.any():
    raise ValueError(
        f"{edo_missing.sum()} Edo questions are missing."
    )

# Metadata shared by both language versions.
metadata_columns = [
    "record_id",
    "translation_group_id",
    "split_group_id",
    "primary_intent",
    "split"
]

# Preserve near-duplicate group where available.
if "near_duplicate_group" in bilingual_df.columns:
    metadata_columns.append("near_duplicate_group")

# English examples.
english_examples = bilingual_df[metadata_columns].copy()

english_examples["example_id"] = (
    english_examples["record_id"] + "_english"
)

english_examples["language"] = "english"

english_examples["question_text"] = (
    bilingual_df["english_question"].astype(str).str.strip()
)

english_examples["translation_status"] = "original_english"

# Edo examples.
edo_examples = bilingual_df[metadata_columns].copy()

edo_examples["example_id"] = (
    edo_examples["record_id"] + "_edo_bini"
)

edo_examples["language"] = "edo_bini"

edo_examples["question_text"] = (
    bilingual_df["edo_question_final"].astype(str).str.strip()
)

edo_examples["translation_status"] = (
    "machine_generated_provisional"
)

# Combine both language versions.
classifier_bilingual_df = pd.concat(
    [english_examples, edo_examples],
    ignore_index=True
)

# Consistent column ordering.
classifier_columns = [
    "example_id",
    "record_id",
    "translation_group_id",
    "split_group_id",
    "language",
    "question_text",
    "primary_intent",
    "split",
    "translation_status"
]

if "near_duplicate_group" in classifier_bilingual_df.columns:
    classifier_columns.append("near_duplicate_group")

classifier_bilingual_df = (
    classifier_bilingual_df[classifier_columns]
)

print("BILINGUAL CLASSIFIER DATASET")
print("----------------------------")

print("Total examples:", len(classifier_bilingual_df))
print("Unique underlying records:",
      classifier_bilingual_df["record_id"].nunique())

print("\nLanguage distribution:")
display(
    classifier_bilingual_df["language"]
    .value_counts()
    .to_frame("examples")
)

print("\nSplit distribution:")
display(
    pd.crosstab(
        classifier_bilingual_df["split"],
        classifier_bilingual_df["language"]
    )
)

print("\nIntent distribution:")
display(
    pd.crosstab(
        classifier_bilingual_df["primary_intent"],
        classifier_bilingual_df["language"]
    )
)

BILINGUAL CLASSIFIER DATASET
----------------------------
Total examples: 612
Unique underlying records: 306

Language distribution:


,examples
language,
english,306
edo_bini,306



Split distribution:


language,edo_bini,english
split,,
test,61,61
train,196,196
validation,49,49



Intent distribution:


language,edo_bini,english
primary_intent,,
crop_cultivation,17,17
disease_management,40,40
fertilizer_application,40,40
harvesting,33,33
irrigation,32,32
out_of_scope,75,75
pest_management,25,25
soil_management,44,44


## 4. Validate bilingual pairing and prevent leakage

Check that:

1. Every underlying record has exactly two examples.
2. Each record has one English and one Edo example.
3. Both versions have the same intent label.
4. Both versions belong to the same dataset split.
5. Every example has a unique example ID.
6. No question text is empty.

In [7]:
# Every record should have exactly two examples.
examples_per_record = (
    classifier_bilingual_df.groupby("record_id").size()
)

if not (examples_per_record == 2).all():
    raise ValueError(
        "Some records do not have exactly two language examples."
    )

# Every record should have both languages.
languages_per_record = (
    classifier_bilingual_df.groupby("record_id")["language"]
    .agg(set)
)

expected_languages = {"english", "edo_bini"}

if not languages_per_record.apply(
    lambda languages: languages == expected_languages
).all():
    raise ValueError(
        "Some records are missing a language version."
    )

# Each record must have one label and one split.
for column in ["primary_intent", "split", "split_group_id"]:
    counts = (
        classifier_bilingual_df.groupby("record_id")[column]
        .nunique()
    )

    if not (counts == 1).all():
        raise ValueError(
            f"Different {column} values found between language versions."
        )

# Unique model example identifiers.
if classifier_bilingual_df["example_id"].duplicated().any():
    raise ValueError("Duplicate example IDs detected.")

# No empty model inputs.
if (
    classifier_bilingual_df["question_text"]
    .astype(str).str.strip().eq("")
).any():
    raise ValueError("Empty classifier input detected.")

# Expected sizes based on the original split.
expected_sizes = {
    "train": 392,
    "validation": 98,
    "test": 122
}

actual_sizes = (
    classifier_bilingual_df["split"]
    .value_counts()
    .to_dict()
)

for split_name, expected_count in expected_sizes.items():
    actual_count = actual_sizes.get(split_name, 0)

    if actual_count != expected_count:
        raise ValueError(
            f"{split_name}: expected {expected_count}, "
            f"got {actual_count}"
        )

print("BILINGUAL VALIDATION PASSED")
print("---------------------------")
print("Total examples:", len(classifier_bilingual_df))
print("Unique records:", classifier_bilingual_df["record_id"].nunique())
print("Unique example IDs:",
      classifier_bilingual_df["example_id"].nunique())
print("Language pairs: complete")
print("Intent consistency: passed")
print("Split consistency: passed")
print("Empty inputs: none")

print("\nFinal split sizes:")
print(
    classifier_bilingual_df["split"].value_counts()
    .reindex(["train", "validation", "test"])
)

BILINGUAL VALIDATION PASSED
---------------------------
Total examples: 612
Unique records: 306
Unique example IDs: 612
Language pairs: complete
Intent consistency: passed
Split consistency: passed
Empty inputs: none

Final split sizes:
split
train         392
validation     98
test          122
Name: count, dtype: int64


## 5. Export the provisional bilingual classifier dataset

Save the complete dataset and separate training, validation, and test
files.

These files will be used in the next notebook for baseline model
training and evaluation.

In [8]:
# Complete bilingual dataset.
full_output = (
    PROCESSED_DIR /
    "classifier_bilingual_dataset_provisional_v1.csv"
)

classifier_bilingual_df.to_csv(
    full_output,
    index=False
)

# Individual split exports.
split_outputs = {}

for split_name in ["train", "validation", "test"]:

    split_df = classifier_bilingual_df[
        classifier_bilingual_df["split"] == split_name
    ].copy()

    output_path = (
        PROCESSED_DIR /
        f"classifier_bilingual_{split_name}_provisional_v1.csv"
    )

    split_df.to_csv(output_path, index=False)

    split_outputs[split_name] = {
        "path": str(output_path),
        "records": len(split_df)
    }

# Export summary.
summary_df = (
    classifier_bilingual_df
    .groupby(["split", "language", "primary_intent"])
    .size()
    .reset_index(name="example_count")
)

summary_path = (
    PROCESSED_DIR /
    "classifier_bilingual_summary_v1.csv"
)

summary_df.to_csv(summary_path, index=False)

print("BILINGUAL EXPORT COMPLETED")
print("--------------------------")

print("Full dataset:", full_output)
print("Total examples:", len(classifier_bilingual_df))

for split_name, details in split_outputs.items():
    print(
        f"{split_name}: {details['records']} examples"
    )
    print("Saved:", details["path"])

print("\nSummary:", summary_path)

BILINGUAL EXPORT COMPLETED
--------------------------
Full dataset: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\classifier_bilingual_dataset_provisional_v1.csv
Total examples: 612
train: 392 examples
Saved: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\classifier_bilingual_train_provisional_v1.csv
validation: 98 examples
Saved: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\classifier_bilingual_validation_provisional_v1.csv
test: 122 examples
Saved: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\classifier_bilingual_test_provisional_v1.csv

Summary: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\classifier_bilingual_summary_v1.csv
